# Atlas-100M-T: a larger traffic adapter on the frozen Atlas-100M (ParadoxAI Lab)

Rules: `atlas/m100/PREREG_TRAFFIC.md`, committed before this ran. Weather and air quality are not touched: the shared core is frozen and only the traffic view is loaded. Run: A100 runtime → Run all → allow Google Drive (the account holding `PWM-100M/`). Training resumes from Drive after a disconnect.

In [ ]:
# 1. Drive and GPU
import os, shutil, hashlib, json, glob
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
STORE = '/content/drive/MyDrive/PWM-100M'; OUT = f'{STORE}/out'; TP = f'{STORE}/traffic_plus'
assert os.path.exists(f'{OUT}/atlas100m_traffic_view.pt'), 'Atlas-100M traffic view missing'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code and data (traffic only)
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!pip install -q einops 2>&1 | tail -1
V06 = '/content/pwm/v0.6-conference'
!cd {V06} && python tools/fetch_all.py 2>&1 | tail -6
for d in ('submissions', 'results/paradoxbench'):                      # every opponent's saved submission
    shutil.copytree(f'{STORE}/v06/{d}', f'{V06}/{d}', dirs_exist_ok=True)
print(len(glob.glob(f'{V06}/submissions/*.npz')), 'submissions restored')

In [ ]:
# 3. Check W (before): the published weather / air-quality / full-model files are byte-identical to SHA256SUMS
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
SUMS = dict(l.split()[::-1] for l in open('/content/pwm/atlas/m100/results/SHA256SUMS'))
W = {f: sha(f'{OUT}/{f}') == SUMS[f] for f in ['atlas100m_best.pt', 'atlas100m_weather_view.pt', 'atlas100m_aq_view.pt']}
print('CHECK W before:', W); assert all(W.values())

In [ ]:
# 4. Train Atlas-100M-T (check T0 inside; resumes from Drive)
!cd {V06} && python -u ../atlas/m100/traffic_plus.py train --base {OUT}/atlas100m_traffic_view.pt --out {TP} 2>&1 | grep --line-buffered -v "Warning\|warn("

In [ ]:
# 5. Score once on the official test windows, paired against every submission (incl. Atlas-100M)
!cd {V06} && python -u ../atlas/m100/score_traffic.py --plus {TP}/atlas100mT_best.pt --name Atlas-100M-T --summary {OUT}/atlas100m_summary.json --out {TP} 2>&1 | grep --line-buffered "ParadoxBench\|TEST\|Error"

In [ ]:
# 6. Save, check W (after), copy results to the shared folder
for d in ('submissions', 'results/paradoxbench'):
    shutil.copytree(f'{V06}/{d}', f'{STORE}/v06/{d}', dirs_exist_ok=True)
W = {f: sha(f'{OUT}/{f}') == SUMS[f] for f in ['atlas100m_best.pt', 'atlas100m_weather_view.pt', 'atlas100m_aq_view.pt']}
print('CHECK W after:', W)
DST = '/content/drive/MyDrive/PWM-10M/m100_out'; os.makedirs(DST, exist_ok=True)
for f in glob.glob(f'{TP}/*.json') + glob.glob(f'{TP}/*.jsonl'): shutil.copy(f, DST)
print(sorted(os.listdir(DST)))